# logit-thermo 探索の続き：C の手の確かめ直し ＋ 候補 A（長さ）の下見（CPU）

- **必要な secret：なし**（`HF_TOKEN` があれば取り込みが少し速くなるだけ）。**ランタイム：CPU（標準）**。見込みは 20〜40 分。
- **探索**です。確かめには使いません。
- 前半（案 1）：探索 C の「写し歩」を決める並びの長さを 3 と 8 に変えて分け直す（ニュースの 9 条件）。A2-3 の 6 条件で、位置ごとのフィードバックを出す。保存先 `A2_explore/more_part1.json`。
- 後半（案 2）：2025-01〜06 の文章で、書き出し 64 ＋ 続き 2,048 トークン以上ある文書の数を、データセット・月ごとに数える。**中身は表示も保存もしません**（数だけ）。保存先 `A2_explore/more_part2.json`。
- 計算のコード：`scripts/a2_2_collect.py`（`e578cfc480980be6…`）、`scripts/a2_explore_copy.py`、`scripts/a2_explore_more.py`（`1ea93fc85d36bef1…`）をそのまま埋め込み。
- **通知**：🚀・🎉・⚠️。名札は「logit-thermo 探索の続き（C の手の確かめ直しと、長い文章の下見、CPU）」。終わったらランタイムを切ります。


In [ ]:
# セル1：通知、GPU、Drive
import os, time, urllib.request
NTFY_TOPIC = 'YOUR_NTFY_TOPIC'
NB_LABEL = 'logit-thermo 探索の続き（C の手の確かめ直しと、長い文章の下見、CPU）'
def notify(msg, tags='pig'):
    try:
        req = urllib.request.Request('https://ntfy.sh/' + NTFY_TOPIC, data=(msg + '\n[' + NB_LABEL + ']').encode('utf-8'),
                                     headers={'Title': 'logit-thermo - explore more', 'Tags': tags})
        urllib.request.urlopen(req, timeout=10).read()
        print('通知を送りました：', msg)
    except Exception as e:
        print('通知は送れませんでした（実験には影響しません）：', type(e).__name__)
_NOTIFIED = {'error': False}
def _on_error(result):
    if result.error_in_exec is not None and not _NOTIFIED['error']:
        _NOTIFIED['error'] = True
        import traceback
        e = result.error_in_exec
        txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
        v = os.environ.get('HF_TOKEN')
        if v: txt = txt.replace(v, '***')
        txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
        notify('⚠️ 止まりました（エラー）：\n' + txt, tags='warning')
if not globals().get('_HOOK_ON'):
    get_ipython().events.register('post_run_cell', _on_error); _HOOK_ON = True
notify('🚀 スタートしました（これが届けば通知の設定は OK です）', tags='rocket')
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    pass

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# セル2：計算のコード
import hashlib
SRC22 = '"""A2-2（ゲート v0.3）のデータ集め。GPU のノートブックにこのファイルの中身をそのまま埋め込む。\n\n何も判定しない（分析は別の CPU ノートブック）。各モデルについて：\n- 人の文脈（teacher forcing）：各位置・各温度の ℓ（裾の取り分の log-odds）、1 位の log-odds、エントロピー。\n- 温度計の材料：β ＝ 1/T の格子ごとに Σ(⟨z⟩_β − z_y)（人のトークンの平均エネルギーとの差）。\n- a_loc・h（T ＝ 1、人の文脈の各位置）。\n- 生成：各温度・各サンプルで 512 歩、各一歩の ℓ・1 位の log-odds・エントロピーと、選んだトークン。\nすべて float64 で計算し、float32 で保存する。使わない番号（tokenizer の長さ以上）と EOS は −∞ にして外す。\n"""\nimport gc, hashlib, json, math, os, re, time\nimport numpy as np\nimport torch\n\nH = 100\nPROMPT, CONT, N_DOCS, SAMPLES = 64, 512, 40, 2\nTEMPS = [0.6, 0.7, 0.8, 0.85, 0.9, 0.95, 1.0, 1.05, 1.1, 1.15, 1.2, 1.25, 1.3, 1.4, 1.5]\nTCAL_T = np.exp(np.linspace(np.log(0.4), np.log(2.5), 61))   # 温度計の格子（T）\nSEED = 20261027\nMODELS = [\n    (\'pythia-1b\', \'EleutherAI/pythia-1b\', \'Pythia\'),\n    (\'qwen2.5-0.5b\', \'Qwen/Qwen2.5-0.5B\', \'Qwen\'),\n    (\'qwen2.5-0.5b-inst\', \'Qwen/Qwen2.5-0.5B-Instruct\', \'Qwen\'),\n    (\'qwen2.5-1.5b\', \'Qwen/Qwen2.5-1.5B\', \'Qwen\'),\n    (\'olmo2-1b\', \'allenai/OLMo-2-0425-1B\', \'OLMo\'),\n    (\'smollm2-1.7b\', \'HuggingFaceTB/SmolLM2-1.7B\', \'SmolLM\'),\n    (\'smollm2-1.7b-inst\', \'HuggingFaceTB/SmolLM2-1.7B-Instruct\', \'SmolLM\'),\n]\nCROSS = [(\'pythia-1b\', \'olmo2-1b\'), (\'qwen2.5-1.5b\', \'smollm2-1.7b\')]\nBBC_MONTHS = [\'2025-04\', \'2025-05\', \'2025-06\']\nBOILER = re.compile(r\'(Getty Images|Image source|Image caption|Related topics|Follow BBC|^Watch:|^Listen:|To play this video you need to enable JavaScript)\')\nH_FRAC = 0.00066   # H を語彙の割合にした版（ゲート §2.1）\n\n\ndef h2_of(n_vocab):\n    return max(2, int(round(H_FRAC * n_vocab)))\n\n\n# ---------- 一歩の量（torch、float64） ----------\n\ndef step_stats(z, T, valid, H2):\n    """z：(B, V) の logit、valid：(V,) の bool。戻り値：ℓ（順位 H 以降）、1 位の log-odds、エントロピー、ℓ（順位 H2 以降）。\n    どれも (B,)、float64。1 位の log-odds は「1 位を除いた logsumexp」から直接計算する（p1 → 1 でも有限）。"""\n    z = z.double().masked_fill(~valid, float(\'-inf\')) / T\n    lse = torch.logsumexp(z, dim=-1)\n    top = torch.topk(z, max(H, H2) - 1, dim=-1)\n\n    def ell_of(h):\n        head = torch.logsumexp(top.values[:, :h - 1], dim=-1)\n        return torch.logsumexp(z.scatter(-1, top.indices[:, :h - 1], float(\'-inf\')), dim=-1) - head\n    rest = torch.logsumexp(z.scatter(-1, top.indices[:, :1], float(\'-inf\')), dim=-1)\n    top1 = top.values[:, 0] - rest\n    lp = z - lse[:, None]\n    p = torch.exp(lp)\n    ent = -torch.where(p > 0, p * lp, torch.zeros_like(p)).sum(-1)\n    return ell_of(H), top1, ent, ell_of(H2)\n\n\ndef np_step_stats(z, T, valid, H2):\n    """自己テスト用の素直な numpy 版（1 行）。"""\n    z = np.where(valid, z.astype(np.float64), -np.inf) / T\n    zs = np.sort(z[np.isfinite(z)])[::-1]\n    def lse(x):\n        m = x.max(); return m + np.log(np.exp(x - m).sum())\n    ell = lse(zs[H - 1:]) - lse(zs[:H - 1])\n    ell2 = lse(zs[H2 - 1:]) - lse(zs[:H2 - 1])\n    lp = zs - lse(zs)\n    return ell, float(zs[0] - lse(zs[1:])), float(-(np.exp(lp) * lp).sum()), ell2\n\n\ndef shape_stats(z, valid):\n    """T ＝ 1 の a_loc（順位 300〜3,000 の傾き）と h（頭の確率 ÷ 裾の確率の log10）。z：(N, V)。"""\n    z = z.double().masked_fill(~valid, float(\'-inf\'))\n    zs = torch.sort(z, dim=-1, descending=True).values\n    r = torch.arange(300, 3001, device=z.device, dtype=torch.float64)\n    x = torch.log(r); x = x - x.mean()\n    y = zs[:, 299:3000]\n    a = -((y - y.mean(-1, keepdim=True)) * x).sum(-1) / (x * x).sum()\n    h = (torch.logsumexp(zs[:, :H - 1], -1) - torch.logsumexp(zs[:, H - 1:], -1)) / math.log(10)\n    return a, h\n\n\ndef tcal_terms(z, y, valid):\n    """温度計の材料：各 T（TCAL_T）で Σ_t(⟨z⟩_T − z_y)。z：(N, V)、y：(N,)。"""\n    z = z.double().masked_fill(~valid, float(\'-inf\'))\n    zy = z.gather(-1, y[:, None])[:, 0]\n    out = []\n    for T in TCAL_T:\n        lp = torch.log_softmax(z / T, dim=-1)\n        p = torch.exp(lp)\n        mean_z = torch.where(p > 0, p * z, torch.zeros_like(p)).sum(-1)\n        out.append(float((mean_z - zy).sum()))\n    return np.array(out)\n\n\n# ---------- 書き出しの文章 ----------\n\ndef clean(text):\n    paras = [p.strip() for p in text.split(\'\\n\')]\n    keep = [p for p in paras if len(p) >= 60 and not BOILER.search(p)]\n    return \'\\n\\n\'.join(keep)\n\n\ndef load_docs():\n    from datasets import load_dataset\n    pool = []\n    for m in BBC_MONTHS:\n        ds = load_dataset(\'RealTimeData/bbc_news_alltime\', m, split=\'train\')\n        for row in ds:\n            c = clean(row[\'content\'] or \'\')\n            if len(c) >= 4000:\n                pool.append({\'month\': m, \'link\': row[\'link\'], \'text\': c})\n    pool.sort(key=lambda d: hashlib.sha256(d[\'link\'].encode()).hexdigest())\n    rng = np.random.default_rng(SEED)\n    idx = rng.choice(len(pool), size=N_DOCS, replace=False)\n    docs = [pool[i] for i in sorted(idx)]\n    meta = {\'pool_size\': len(pool), \'docs\': [{\'month\': d[\'month\'], \'link_sha256\': hashlib.sha256(d[\'link\'].encode()).hexdigest(),\n                                               \'text_sha256\': hashlib.sha256(d[\'text\'].encode()).hexdigest(),\n                                               \'chars\': len(d[\'text\'])} for d in docs]}\n    return docs, meta\n\n\n# ---------- モデルごと ----------\n\ndef load_model(hf_id, device):\n    from transformers import AutoModelForCausalLM, AutoTokenizer\n    tok = AutoTokenizer.from_pretrained(hf_id)\n    model = AutoModelForCausalLM.from_pretrained(hf_id, torch_dtype=torch.float32).to(device).eval()\n    return tok, model\n\n\ndef valid_mask(tok, model, device):\n    V = model.get_output_embeddings().weight.shape[0]\n    valid = torch.zeros(V, dtype=torch.bool, device=device)\n    valid[:len(tok)] = True\n    eos = set()\n    if tok.eos_token_id is not None:\n        eos.add(int(tok.eos_token_id))\n    ge = getattr(model, \'generation_config\', None)\n    if ge is not None and ge.eos_token_id is not None:\n        eos |= set([ge.eos_token_id] if isinstance(ge.eos_token_id, int) else ge.eos_token_id)\n    eot = tok.convert_tokens_to_ids(\'<|endoftext|>\') if \'<|endoftext|>\' in tok.get_vocab() else None\n    if eot is not None:\n        eos.add(int(eot))   # 文書の終わりの記号も外す（SmolLM2-Instruct は EOS が <|im_end|> なので、ここで足す）\n    for e in eos:\n        if e < V:\n            valid[e] = False\n    return valid, sorted(eos)\n\n\ndef split_doc(tok, text):\n    ids = tok(text, add_special_tokens=False).input_ids\n    bos = [tok.bos_token_id] if (tok.bos_token_id is not None and getattr(tok, \'add_bos_token\', False)) else []\n    if len(ids) < PROMPT + CONT:\n        return None\n    return bos + ids[:PROMPT], ids[PROMPT:PROMPT + CONT]\n\n\n@torch.no_grad()\ndef teacher_forced(model, prompt, human, valid, device, H2):\n    x = torch.tensor([prompt + human], device=device)\n    logits = model(x).logits[0, len(prompt) - 1: len(prompt) - 1 + CONT].float()\n    y = torch.tensor(human, device=device)\n    res = {}\n    for k, T in enumerate(TEMPS):\n        res[k] = [v.float().cpu().numpy() for v in step_stats(logits, T, valid, H2)]\n    a, h = shape_stats(logits, valid)\n    return res, a.float().cpu().numpy(), h.float().cpu().numpy(), tcal_terms(logits, y, valid)\n\n\n@torch.no_grad()\ndef generate(model, prompts, T, valid, gen, H2):\n    """prompts：同じ長さのトークン列のリスト。512 歩を KV キャッシュで書く。"""\n    x = torch.tensor(prompts, device=valid.device)\n    out = model(x, use_cache=True)\n    past, logits = out.past_key_values, out.logits[:, -1, :].float()\n    L, T1, E, L2, toks = [], [], [], [], []\n    for t in range(CONT):\n        ell, top1, ent, ell2 = step_stats(logits, T, valid, H2)\n        L.append(ell); T1.append(top1); E.append(ent); L2.append(ell2)\n        z = logits.double().masked_fill(~valid, float(\'-inf\')) / T\n        p = torch.softmax(z, dim=-1)\n        nxt = torch.multinomial(p, 1, generator=gen)\n        toks.append(nxt[:, 0])\n        out = model(nxt, past_key_values=past, use_cache=True)\n        past, logits = out.past_key_values, out.logits[:, -1, :].float()\n    st = lambda xs: torch.stack(xs, 1).float().cpu().numpy()\n    return st(L), st(T1), st(E), st(L2), torch.stack(toks, 1).int().cpu().numpy()\n\n\ndef run_model(name, hf_id, docs, outdir, device, notify=print):\n    path = f\'{outdir}/{name}.npz\'\n    if os.path.exists(path):\n        print(name, \'はもう終わっています（飛ばします）\')\n        return\n    t0 = time.time()\n    tok, model = load_model(hf_id, device)\n    valid, eos = valid_mask(tok, model, device)\n    H2 = h2_of(len(tok))\n    pairs = [split_doc(tok, d[\'text\']) for d in docs]\n    use = [i for i, p in enumerate(pairs) if p is not None]\n    nT, nD = len(TEMPS), len(use)\n    tf = np.zeros((4, nD, nT, CONT), np.float32)   # [ℓ(H), 1 位の log-odds, エントロピー, ℓ(H2)]\n    a_all = np.zeros((nD, CONT), np.float32); h_all = np.zeros((nD, CONT), np.float32)\n    tcal = np.zeros((nD, len(TCAL_T)))\n    for j, i in enumerate(use):\n        res, a, h, tc = teacher_forced(model, *pairs[i], valid, device, H2)\n        for k in range(nT):\n            for q in range(4):\n                tf[q, j, k] = res[k][q]\n        a_all[j], h_all[j], tcal[j] = a, h, tc\n    plen = {len(pairs[i][0]) for i in use}\n    assert len(plen) == 1, plen\n    prompts = [pairs[i][0] for i in use]\n    gen = np.zeros((4, nT, SAMPLES, nD, CONT), np.float32)\n    toks = np.zeros((nT, SAMPLES, nD, CONT), np.int32)\n    for k, T in enumerate(TEMPS):\n        for s in range(SAMPLES):\n            g = torch.Generator(device=device).manual_seed(SEED + 1000 * k + s)\n            l, t1, e, l2, tk = generate(model, prompts, T, valid, g, H2)\n            gen[0, k, s], gen[1, k, s], gen[2, k, s], gen[3, k, s], toks[k, s] = l, t1, e, l2, tk\n        print(f\'  {name} T={T} おわり（{time.time() - t0:.0f} 秒）\')\n    np.savez_compressed(path + \'.part.npz\', tf=tf, a=a_all, h=h_all, tcal=tcal, tcal_T=TCAL_T, gen=gen, toks=toks,\n                        temps=np.array(TEMPS), use=np.array(use), eos=np.array(eos), tok_len=len(tok),\n                        prompt_len=list(plen)[0], revision=str(getattr(model.config, \'_commit_hash\', \'\')),\n                        seconds=time.time() - t0, H2=H2)\n    os.replace(path + \'.part.npz\', path)\n    del model; gc.collect(); torch.cuda.empty_cache()\n    notify(f\'{name} おわり（{time.time() - t0:.0f} 秒）\')\n\n\n@torch.no_grad()\ndef cross_model(reader, writer, docs, outdir, device):\n    """reader が writer の書いた文を読む。文字の窓で、人の続きを読んだときと比べる（記録用）。"""\n    path = f\'{outdir}/cross_{reader}_reads_{writer}.npz\'\n    if os.path.exists(path):\n        return\n    from transformers import AutoTokenizer\n    hf = dict((n, h) for n, h, _ in MODELS)\n    wtok = AutoTokenizer.from_pretrained(hf[writer])\n    W = np.load(f\'{outdir}/{writer}.npz\')\n    tok, model = load_model(hf[reader], device)\n    valid, _ = valid_mask(tok, model, device)\n    H2 = h2_of(len(tok))\n    use = list(W[\'use\']); Wt = W[\'toks\']\n    out = np.full((2, len(TEMPS), SAMPLES, len(use)), np.nan)   # [cross, human] の ℓ の平均\n    for j, i in enumerate(use):\n        text = docs[i][\'text\']\n        enc = wtok(text, add_special_tokens=False, return_offsets_mapping=True)\n        pend = enc[\'offset_mapping\'][PROMPT - 1][1]                  # 書き手の書き出しの終わりの文字位置\n        pre = text[:pend]\n        for k, T in enumerate(TEMPS):\n            for s in range(SAMPLES):\n                gtxt = wtok.decode(Wt[k, s, j])\n                C = min(len(gtxt), len(text) - pend)\n                for q, cont in enumerate([gtxt[:C], text[pend:pend + C]]):\n                    e2 = tok(pre + cont, add_special_tokens=False, return_offsets_mapping=True)\n                    ids = e2[\'input_ids\']; offs = e2[\'offset_mapping\']\n                    bos = [tok.bos_token_id] if (tok.bos_token_id is not None and getattr(tok, \'add_bos_token\', False)) else []\n                    x = torch.tensor([bos + ids], device=device)\n                    logits = model(x).logits[0].float()\n                    pos = [n for n, (a0, _) in enumerate(offs) if pend <= a0 < pend + C and n > 0]\n                    if not pos:\n                        continue\n                    pred = torch.tensor([n - 1 + len(bos) for n in pos], device=device)\n                    ell = step_stats(logits[pred], T, valid, H2)[0]\n                    out[q, k, s, j] = float(ell.mean())\n    np.savez_compressed(path + \'.part.npz\', out=out, temps=np.array(TEMPS))\n    os.replace(path + \'.part.npz\', path)\n    del model; gc.collect(); torch.cuda.empty_cache()\n\n\n# ---------- 自己テスト（本番の前に） ----------\n\n@torch.no_grad()\ndef self_test(device):\n    from transformers import AutoModelForCausalLM, AutoTokenizer\n    hf_id = \'hf-internal-testing/tiny-random-GPT2LMHeadModel\'\n    tok = AutoTokenizer.from_pretrained(hf_id)\n    model = AutoModelForCausalLM.from_pretrained(hf_id, torch_dtype=torch.float32).to(device).eval()\n    valid, _ = valid_mask(tok, model, device)\n    H2 = 33\n    # ① 一歩の量が numpy の素直な計算と一致する（1 位が飛び抜けた行も入れる：1 位の log-odds が有限か）\n    z = torch.randn(3, valid.shape[0], device=device) * 3\n    z[2, 7] += 80.0\n    for T in [0.6, 1.0, 1.5]:\n        got_all = step_stats(z, T, valid, H2)\n        assert all(bool(torch.isfinite(v).all()) for v in got_all), (\'有限でない値\', T)\n        for b in range(3):\n            ref = np_step_stats(z[b].cpu().numpy(), T, valid.cpu().numpy(), H2)\n            got = tuple(float(v[b]) for v in got_all)\n            assert all(abs(g - r) < 1e-6 * max(1, abs(r)) for g, r in zip(got, ref)), (T, got, ref)\n    # ② KV キャッシュの生成と、まとめて計算した logit が一致する\n    g = torch.Generator(device=device).manual_seed(0)\n    prompts = [list(range(5, 5 + 16)), list(range(30, 30 + 16))]\n    x = torch.tensor(prompts, device=device)\n    out = model(x, use_cache=True); past, lg = out.past_key_values, out.logits[:, -1, :]\n    seq, steps = x, []\n    for t in range(8):\n        nxt = torch.multinomial(torch.softmax(lg.masked_fill(~valid, float(\'-inf\')), -1), 1, generator=g)\n        steps.append(lg.clone()); seq = torch.cat([seq, nxt], 1)\n        out = model(nxt, past_key_values=past, use_cache=True); past, lg = out.past_key_values, out.logits[:, -1, :]\n    full = model(seq).logits\n    for t in range(8):\n        assert torch.allclose(full[:, 15 + t, :], steps[t], atol=1e-4), (\'KV キャッシュの不一致\', t)\n    # ③ 1 歩目は生成と人の文脈で同じ分布\n    human = list(range(100, 100 + 64))   # 小さなモデルは位置 512 までなので短くする（使うのは位置 15 だけ）\n    tf_logits = model(torch.tensor([prompts[0] + human], device=device)).logits[0, 15]\n    gen_first = model(torch.tensor([prompts[0]], device=device)).logits[0, -1]\n    assert torch.allclose(tf_logits, gen_first, atol=1e-5), \'1 歩目が一致しない\'\n    return \'OK\'\n'
SRCX = '"""探索：書き進めたあとに裾が細るのは「結晶」（写し歩）か「ガラス」（写していない歩でも自信が固まる）か。\ndocs/A2_EXPLORE_T0_FEATURES.md §5 の候補 C の「手」。A2-2・A2-3 の生のファイルを見直す探索で、確かめには使わない。\n\nscripts/a2_2_collect.py を実行したのと同じ名前空間で実行する（load_docs・split_doc・step_stats・valid_mask・h2_of・\nload_model・TEMPS・PROMPT・CONT を使う）。CPU だけで動く（モデルを読むのは検量線の確かめの Qwen2.5-0.5B だけ）。\n\n写し歩：その歩のトークンで終わる 4 トークンの並びが、それより前（書き出しを含む）にすでに出てきている歩。\n新しい歩：それ以外。\n\n位置の帯ごとに、記事ごと（生成は 2 本をまとめる）に次の分け方をする。分け方は恒等式（記事ごとに足すと f に一致する）：\n  f ＝ ガラスの分 ＋ 結晶の分\n  ガラスの分 ＝ ℓ_gen（新しい歩の平均）− ℓ_tf（人の文の新しい歩の平均）\n  結晶の分   ＝ c_gen ×（ℓ_gen（写し歩）− ℓ_gen（新しい歩））− c_hum ×（ℓ_tf（写し歩）− ℓ_tf（新しい歩））\n  （c は写し歩の割合。写し歩がない記事ではその項は 0。）\n\n読み方（データを開く前に決める。探索の目安で、判定ではない）：\n- 見るところ：温度 0.7 と 0.8、位置の帯 128〜512、記事の平均で f ≤ −0.2 のとき。\n- ガラスの割合 ＝ ガラスの分 ÷ f。0.5 以上なら「ガラス寄り」、0.25 以下なら「結晶寄り」、その間は「混ざり」。\n- 検量線（人工の結晶）：人の続きの前半 256 トークンを、後半にそのまま写した文を Qwen2.5-0.5B に読ませる（記事 10 本）。\n  元の人の文と比べて、帯 128〜512 で写し歩の割合が 0.4 以上、かつガラスの割合が 0.5 未満（結晶の分が半分以上）なら\n  「手は結晶を結晶と言える」。通らなければ、この手で「結晶」「ガラス」という言葉は使わない。\n  ガラスの人工の標準は作れていない（限界として書く）。\n"""\nimport hashlib, json, math, time\nimport numpy as np\n\nN_GRAM = 4\nBINS = [(1, 8), (8, 32), (32, 128), (128, 512)]\nLOOK_T = [0.7, 0.8]\nCAL_DOCS, CAL_HALF = 10, 256\nCAL_MODEL = (\'qwen2.5-0.5b\', \'Qwen/Qwen2.5-0.5B\')\nBBC_CONDITIONS = [  # (名前, npz のパス（フォルダからの相対）, tokenizer の Hugging Face の ID)\n    (\'pythia-1b\', \'A2_2/pythia-1b.npz\', \'EleutherAI/pythia-1b\'),\n    (\'qwen2.5-0.5b\', \'A2_2/qwen2.5-0.5b.npz\', \'Qwen/Qwen2.5-0.5B\'),\n    (\'qwen2.5-0.5b-inst\', \'A2_2/qwen2.5-0.5b-inst.npz\', \'Qwen/Qwen2.5-0.5B-Instruct\'),\n    (\'qwen2.5-1.5b\', \'A2_2/qwen2.5-1.5b.npz\', \'Qwen/Qwen2.5-1.5B\'),\n    (\'olmo2-1b\', \'A2_2/olmo2-1b.npz\', \'allenai/OLMo-2-0425-1B\'),\n    (\'smollm2-1.7b\', \'A2_2/smollm2-1.7b.npz\', \'HuggingFaceTB/SmolLM2-1.7B\'),\n    (\'smollm2-1.7b-inst\', \'A2_2/smollm2-1.7b-inst.npz\', \'HuggingFaceTB/SmolLM2-1.7B-Instruct\'),\n    (\'qwen2.5-7b@bbc\', \'A2_3/qwen2.5-7b@bbc.npz\', \'Qwen/Qwen2.5-7B\'),\n    (\'olmo2-7b@bbc\', \'A2_3/olmo2-7b@bbc.npz\', \'allenai/OLMo-2-1124-7B\'),\n]\n\n\ndef copy_flags(hist, cont, n=N_GRAM):\n    """hist：書き出しのトークン、cont：続き（長さ L）→ (L,) の bool。続きの各歩が写し歩か。"""\n    seq = list(hist) + list(cont)\n    P = len(hist)\n    seen, flags = set(), np.zeros(len(cont), bool)\n    for i in range(n - 1, len(seq)):\n        g = tuple(seq[i - n + 1:i + 1])\n        if i >= P:\n            flags[i - P] = g in seen\n        seen.add(g)\n    return flags\n\n\ndef split_means(ell, flags):\n    """ell・flags：同じ形の配列（まとめる歩）→（新しい歩の平均, 写し歩の平均, 写し歩の割合, 全体の平均）。"""\n    ell = np.asarray(ell, np.float64).ravel(); flags = np.asarray(flags, bool).ravel()\n    c = float(flags.mean()) if len(flags) else float(\'nan\')\n    new = float(ell[~flags].mean()) if (~flags).any() else float(\'nan\')\n    cp = float(ell[flags].mean()) if flags.any() else float(\'nan\')\n    return new, cp, c, float(ell.mean())\n\n\ndef decompose(gen_ell, gen_flags, tf_ell, tf_flags):\n    """1 記事・1 温度・1 帯。gen_*：(S, 帯の長さ)、tf_*：(帯の長さ,)。→ dict（f・ガラス・結晶・写しの割合）。"""\n    gn, gc, cg, gm = split_means(gen_ell, gen_flags)\n    tn, tc, ch, tm = split_means(tf_ell, tf_flags)\n    if not (np.isfinite(gn) and np.isfinite(tn)):\n        return None  # 新しい歩がない（全部写し）。この記事・帯は外す\n    glass = gn - tn\n    crystal = (cg * (gc - gn) if cg > 0 else 0.0) - (ch * (tc - tn) if ch > 0 else 0.0)\n    f = gm - tm\n    assert abs(glass + crystal - f) < 1e-9, (glass, crystal, f)\n    return dict(f=f, glass=glass, crystal=crystal, c_gen=cg, c_hum=ch)\n\n\ndef summarize(rows):\n    """記事ごとの dict のリスト → 平均（分け方は平均で足し算が成り立つ）と、f の中央値。"""\n    rows = [r for r in rows if r is not None]\n    if not rows:\n        return None\n    out = {k: float(np.mean([r[k] for r in rows])) for k in [\'f\', \'glass\', \'crystal\', \'c_gen\', \'c_hum\']}\n    out[\'f_median\'] = float(np.median([r[\'f\'] for r in rows]))\n    out[\'n_docs\'] = len(rows)\n    return out\n\n\ndef reading(s):\n    if s is None or s[\'f\'] > -0.2:\n        return \'（f が −0.2 より上なので読まない）\'\n    share = s[\'glass\'] / s[\'f\']\n    return \'ガラス寄り\' if share >= 0.5 else \'結晶寄り\' if share <= 0.25 else \'混ざり\'\n\n\ndef analyze_condition(Z, prompts, humans, temps):\n    """Z：npz、prompts・humans：使った記事（use の順）のトークン列。→ {T: {帯: 要約}}。"""\n    toks, gen, tf = Z[\'toks\'], Z[\'gen\'][0], Z[\'tf\'][0]          # (nT,S,nD,L)、(nT,S,nD,L)、(nD,nT,L)\n    nT, S, nD, L = toks.shape\n    hum_flags = [copy_flags(prompts[j], humans[j]) for j in range(nD)]\n    res = {}\n    for k, T in enumerate(temps):\n        gflags = np.zeros((S, nD, L), bool)\n        for s in range(S):\n            for j in range(nD):\n                gflags[s, j] = copy_flags(prompts[j], toks[k, s, j])\n        res[T] = {}\n        for a, b in BINS:\n            rows = [decompose(gen[k, :, j, a:b], gflags[:, j, a:b], tf[j, k, a:b], hum_flags[j][a:b]) for j in range(nD)]\n            res[T][f\'{a}-{b}\'] = summarize(rows)\n    return res\n\n\ndef calibration(docs, device=\'cpu\'):\n    """人工の結晶：人の続きの前半 CAL_HALF トークンを後半に写した文。Qwen2.5-0.5B が読む。"""\n    import torch\n    name, hf_id = CAL_MODEL\n    tok, model = load_model(hf_id, device)\n    valid, _ = valid_mask(tok, model, device)\n    H2 = h2_of(len(tok))\n    res = {T: {f\'{a}-{b}\': [] for a, b in BINS} for T in [0.7, 1.0]}\n    used = 0\n    for d in docs:\n        p = split_doc(tok, d[\'text\'])\n        if p is None:\n            continue\n        prompt, human = p\n        art = human[:CAL_HALF] + human[:CAL_HALF]\n        ells = {}\n        with torch.no_grad():\n            for key, cont in [(\'art\', art), (\'hum\', human)]:\n                x = torch.tensor([prompt + cont], device=device)\n                lg = model(x).logits[0, len(prompt) - 1: len(prompt) - 1 + CONT].float()\n                ells[key] = {T: step_stats(lg, T, valid, H2)[0].double().cpu().numpy() for T in [0.7, 1.0]}\n        fa, fh = copy_flags(prompt, art), copy_flags(prompt, human)\n        for T in [0.7, 1.0]:\n            for a, b in BINS:\n                res[T][f\'{a}-{b}\'].append(decompose(ells[\'art\'][T][None, a:b], fa[None, a:b], ells[\'hum\'][T][a:b], fh[a:b]))\n        used += 1\n        if used >= CAL_DOCS:\n            break\n    out = {T: {k: summarize(v) for k, v in bins.items()} for T, bins in res.items()}\n    s = out[1.0][\'128-512\']\n    ok = bool(s is not None and s[\'c_gen\'] >= 0.4 and s[\'f\'] < 0 and s[\'glass\'] / s[\'f\'] < 0.5)\n    return dict(summary=out, passed=ok, n_docs=used)\n\n\ndef run_all(base, docs, temps):\n    """base：MyDrive/logit-thermo。→ dict。"""\n    from transformers import AutoTokenizer\n    out = {\'calibration\': calibration(docs), \'conditions\': {}}\n    for name, rel, hf_id in BBC_CONDITIONS:\n        t0 = time.time()\n        Z = np.load(f\'{base}/{rel}\')\n        tok = AutoTokenizer.from_pretrained(hf_id)\n        use = [int(i) for i in Z[\'use\']]\n        pairs = [split_doc(tok, docs[i][\'text\']) for i in use]\n        assert all(p is not None for p in pairs)\n        assert len(pairs[0][0]) == int(Z[\'prompt_len\']), (name, len(pairs[0][0]), int(Z[\'prompt_len\']))\n        prompts = [p[0] for p in pairs]; humans = [p[1] for p in pairs]\n        res = analyze_condition(Z, prompts, humans, temps)\n        look = {str(T): reading(res[T][\'128-512\']) for T in LOOK_T}\n        out[\'conditions\'][name] = dict(by_T={str(T): v for T, v in res.items()}, reading=look, seconds=round(time.time() - t0, 1))\n        print(name, look, f\'{time.time() - t0:.0f} 秒\')\n    return out\n\n\ndef self_test():\n    """torch も Hugging Face も使わない確かめ。"""\n    # 写し歩の判定\n    assert copy_flags([1, 2, 3], [4, 1, 2, 3, 4, 5]).tolist() == [False, False, False, False, True, False]\n    assert copy_flags([], [7, 7, 7, 7, 7]).tolist() == [False, False, False, False, True]\n    # 分け方が恒等式（足すと f）になる\n    rng = np.random.default_rng(0)\n    for _ in range(200):\n        L = int(rng.integers(5, 40))\n        ge = rng.normal(size=(2, L)); gf = rng.random((2, L)) < rng.random()\n        te = rng.normal(size=L); tfl = rng.random(L) < rng.random()\n        r = decompose(ge, gf, te, tfl)\n        if r is not None:\n            assert abs(r[\'glass\'] + r[\'crystal\'] - r[\'f\']) < 1e-9\n    # 作り物：写し歩だけ自信が強い（ℓ が低い）→ 結晶寄り。新しい歩でも ℓ が低い → ガラス寄り\n    L = 384\n    gf = np.zeros((2, L), bool); gf[:, 128:] = True\n    tfl = np.zeros(L, bool)\n    te = np.zeros(L)\n    ge_crystal = np.where(gf, -3.0, 0.0)\n    ge_glass = np.full((2, L), -1.0)\n    rc = decompose(ge_crystal, gf, te, tfl); rg = decompose(ge_glass, np.zeros((2, L), bool), te, tfl)\n    assert reading({**rc, \'f_median\': rc[\'f\'], \'n_docs\': 1}) == \'結晶寄り\', rc\n    assert reading({**rg, \'f_median\': rg[\'f\'], \'n_docs\': 1}) == \'ガラス寄り\', rg\n    return \'OK\'\n\n\nif False:\n    print(self_test())\n'
SRCM = '"""探索の続き（2026-10-28 夜）。CPU だけ。確かめには使わない。\n\nscripts/a2_2_collect.py と scripts/a2_explore_copy.py を実行したのと同じ名前空間で実行する\n（load_docs・split_doc・copy_flags・decompose・summarize・reading・BINS・LOOK_T・BBC_CONDITIONS を使う）。\n\npart1(base)：探索 C の手を確かめ直す。\n  (a) 写し歩を決める並びの長さを 3 と 8 に変えて、ガラスの割合を出し直す（ニュースの 9 条件。検量線はしない）。\n      読み方は探索 C と同じ（温度 0.7・0.8、128〜512 トークン目、f ≤ −0.2 のときだけ。ガラスの割合 0.5 以上で\n      「ガラス寄り」、0.25 以下で「結晶寄り」）。4 のときの読み方と、どれだけ食い違うかを数える。\n  (b) A2-3 の 6 条件で、位置の帯（1〜8・8〜32・32〜128・128〜512）ごとの f（記事の中央値）と、帯ごとに f が 0 を\n      横切る温度（ε なしの単純な横切り）を出す（探索 §2 と同じ計算）。\npart2()：候補 A（生成の長さ）の下見。2025-01〜2025-06 の文章で、書き出し 64 ＋ 続き 2,048 トークン（Qwen2.5 の\n  トークナイザーで数える）以上ある文書が何本あるかを、データセット・月ごとに数える。中身は表示しない・保存しない。\n  データセットの読み込みに失敗したら、その理由を記録して次へ進む。\n"""\nimport json, math, time\nimport numpy as np\n\nLONG_MONTHS = [\'2025-01\', \'2025-02\', \'2025-03\', \'2025-04\', \'2025-05\', \'2025-06\']\nLONG_SOURCES = [\'RealTimeData/bbc_news_alltime\', \'RealTimeData/wikitext_alltime\', \'RealTimeData/arxiv_alltime\']\nLONG_NEED = 64 + 2048\nLONG_TOKENIZER = \'Qwen/Qwen2.5-1.5B\'\nA23 = [\'qwen2.5-1.5b@bbc-a100\', \'qwen2.5-7b@bbc\', \'olmo2-7b@bbc\', \'qwen2.5-1.5b@code\', \'smollm2-1.7b@code\', \'pythia-1b@code\']\n\n\ndef analyze_condition_n(Z, prompts, humans, temps, n):\n    """a2_explore_copy.analyze_condition と同じ計算を、写しの並びの長さ n で。"""\n    toks, gen, tf = Z[\'toks\'], Z[\'gen\'][0], Z[\'tf\'][0]\n    nT, S, nD, L = toks.shape\n    hum = [copy_flags(prompts[j], humans[j], n) for j in range(nD)]\n    res = {}\n    for k, T in enumerate(temps):\n        gf = np.zeros((S, nD, L), bool)\n        for s in range(S):\n            for j in range(nD):\n                gf[s, j] = copy_flags(prompts[j], toks[k, s, j], n)\n        res[T] = {f\'{a}-{b}\': summarize([decompose(gen[k, :, j, a:b], gf[:, j, a:b], tf[j, k, a:b], hum[j][a:b])\n                                         for j in range(nD)]) for a, b in BINS}\n    return res\n\n\ndef simple_cross(y, temps):\n    lt = np.log(np.asarray(temps)); y = np.asarray(y, float)\n    for i in range(len(y) - 1):\n        if y[i] < 0 <= y[i + 1]:\n            return float(math.exp(lt[i] + (0 - y[i]) / (y[i + 1] - y[i]) * (lt[i + 1] - lt[i])))\n    return float(\'nan\')\n\n\ndef part1(base, docs):\n    from transformers import AutoTokenizer\n    out = {\'copy_n\': {}, \'a23_positions\': {}}\n    for name, rel, hf_id in BBC_CONDITIONS:\n        t0 = time.time()\n        Zr = np.load(f\'{base}/{rel}\')\n        Z = {k: Zr[k] for k in [\'toks\', \'gen\', \'tf\', \'use\', \'temps\', \'prompt_len\']}\n        temps = [float(t) for t in Z[\'temps\']]\n        tok = AutoTokenizer.from_pretrained(hf_id)\n        pairs = [split_doc(tok, docs[int(i)][\'text\']) for i in Z[\'use\']]\n        assert len(pairs[0][0]) == int(Z[\'prompt_len\'])\n        prompts = [p[0] for p in pairs]; humans = [p[1] for p in pairs]\n        row = {}\n        for n in [3, 8]:\n            res = analyze_condition_n(Z, prompts, humans, temps, n)\n            row[f\'n{n}\'] = dict(by_T={str(T): res[T] for T in temps}, reading={str(T): reading(res[T][\'128-512\']) for T in LOOK_T})\n        out[\'copy_n\'][name] = row\n        print(name, {k: v[\'reading\'] for k, v in row.items()}, f\'{time.time() - t0:.0f} 秒\')\n    for name in A23:\n        Zr = np.load(f\'{base}/A2_3/{name}.npz\')\n        temps = [float(t) for t in Zr[\'temps\']]\n        g = Zr[\'gen\'][0].astype(np.float64).mean(1).transpose(1, 0, 2) - Zr[\'tf\'][0].astype(np.float64)   # (nD, nT, L)\n        bins = {f\'{a}-{b}\': np.median(g[:, :, a:b].mean(-1), 0) for a, b in BINS}\n        out[\'a23_positions\'][name] = {k: dict(f=v.tolist(), cross=simple_cross(v, temps)) for k, v in bins.items()}\n    return out\n\n\ndef part2():\n    from datasets import load_dataset\n    from transformers import AutoTokenizer\n    tok = AutoTokenizer.from_pretrained(LONG_TOKENIZER)\n    out = {}\n    for src in LONG_SOURCES:\n        out[src] = {}\n        for m in LONG_MONTHS:\n            t0 = time.time()\n            try:\n                ds = load_dataset(src, m, split=\'train\')\n                cols = [c for c in ds.column_names if getattr(ds.features[c], \'dtype\', None) == \'string\']\n                field = max(cols, key=lambda c: np.mean([len(x or \'\') for x in ds[c][:50]]))   # いちばん長い文字の列を本文とみなす\n                texts = ds[field]\n                chars = np.array([len(x or \'\') for x in texts])\n                cand = [x for x in texts if x and len(x) >= 6000][:2000]                           # 2,112 トークンの見込みに届くものだけ、最大 2,000 本数える\n                ntok = np.array([len(tok(x, add_special_tokens=False).input_ids) for x in cand]) if cand else np.array([])\n                out[src][m] = dict(field=field, n_docs=int(len(texts)), n_ge_6000_chars=int((chars >= 6000).sum()), capped=bool((chars >= 6000).sum() > 2000), chars_median=float(np.median(chars)) if len(chars) else None,\n                                   n_ge_need=int((ntok >= LONG_NEED).sum()), n_ge_1088=int((ntok >= 64 + 1024).sum()),\n                                   ntok_quartiles=[float(np.percentile(ntok, q)) for q in (25, 50, 75)] if len(ntok) else None,\n                                   seconds=round(time.time() - t0, 1))\n            except Exception as e:\n                out[src][m] = dict(error=f\'{type(e).__name__}: {str(e)[:300]}\')\n            print(src, m, {k: v for k, v in out[src][m].items() if k != \'ntok_quartiles\'})\n    return out\n'
assert hashlib.sha256(SRC22.encode()).hexdigest() == 'e578cfc480980be6b0d1e18377b6beb8b7555e0744f56b13f63cb304a9f73d53'
assert hashlib.sha256(SRCX.encode()).hexdigest() == '012c6dc1fb2ac8a3d120fa2c35a14480c7dc82815e6775cb3829a8bf16284759'
assert hashlib.sha256(SRCM.encode()).hexdigest() == '1ea93fc85d36bef1fec581676720ae7fe675d62cba1cdf5193213f5f3d3b15e1'
exec(SRC22)
exec(SRCX)
exec(SRCM)
print('自己テスト（写しの判定と分け方）：', self_test())


In [ ]:
# セル3：前半（案 1）。記事を読み直して（A2-2 と同じか確かめる）、計算して保存
import json, os
os.makedirs('/content/drive/MyDrive/logit-thermo/A2_explore', exist_ok=True)
docs, meta = load_docs()
old = json.load(open('/content/drive/MyDrive/logit-thermo' + '/A2_2/docs_meta.json'))
assert [d['text_sha256'] for d in old['docs']] == [d['text_sha256'] for d in meta['docs']], 'BBC の記事が A2-2 と違います'
r1 = part1('/content/drive/MyDrive/logit-thermo', docs)
r1['a2_explore_more_sha256'] = '1ea93fc85d36bef1fec581676720ae7fe675d62cba1cdf5193213f5f3d3b15e1'
json.dump(r1, open('/content/drive/MyDrive/logit-thermo/A2_explore' + '/more_part1.json', 'w'), ensure_ascii=False, separators=(',', ':'))
notify('✅ 前半（C の手の確かめ直し）を保存しました', tags='white_check_mark')


In [ ]:
# セル4：後半（案 2）。長い文章の数を数えて保存（中身は保存しない）
r2 = part2()
r2_meta = {'a2_explore_more_sha256': '1ea93fc85d36bef1fec581676720ae7fe675d62cba1cdf5193213f5f3d3b15e1', 'need_tokens': LONG_NEED, 'tokenizer': LONG_TOKENIZER}
json.dump({'counts': r2, 'meta': r2_meta}, open('/content/drive/MyDrive/logit-thermo/A2_explore' + '/more_part2.json', 'w'), ensure_ascii=False, separators=(',', ':'))
print('保存しました')


In [ ]:
# 最後：終わりの通知と、ランタイムを切る
notify('🎉 終わりました：前半と後半を A2_explore に保存しました', tags='tada')
if globals().get('AUTO_DISCONNECT', True):
    from google.colab import drive as _drive, runtime as _runtime
    _drive.flush_and_unmount(); time.sleep(3); _runtime.unassign()
